# TBB · RAG ile Kurumsal Yapay Zekâ Uygulamaları — 1. Gün Uygulamaları
**Temeller: embedding, ingestion, vektör arama**

Bu notebook'u eğitmen kendi ekranında çalıştırır; katılımcılar izler ve tahminleriyle katılır.
Her **CANLI DEMO** bölümü, sunudaki aynı numaralı demo slaytına karşılık gelir.

| Bölüm | Sunudaki yeri | Süre |
| --- | --- | --- |
| Hazırlık | Eğitimden önce | 5 dk |
| CANLI DEMO 1.1 — Token ve embedding | M2 sonunda | 10 dk |
| CANLI DEMO 1.2 — Gerçek mevzuat PDF'i ve chunking | M3 sonunda | 10 dk |
| UYGULAMA 1 — Erdağı Bank indeksleme ve arama | Gün sonu | 40 dk |
| Bonus — HNSW ile hızlı arama | Zaman kalırsa | 5 dk |

*Erdağı Bank ve dokümanları tamamen kurgusaldır. Dokümanlar GitHub reposundaki PDF'lerden indirilir; mevzuat metinleri mevzuat.gov.tr'den.*

## Hazırlık — eğitim başlamadan en az 30 dakika önce

1. Menüden **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçin.
2. Bu bölümdeki hücreleri sırayla çalıştırın (her hücrenin solundaki ▶ düğmesi veya Shift+Enter).
3. Her hücrenin altında hata olmadığını ve son satırda "hazır / yüklendi" mesajını gördüğünüzü kontrol edin.
4. Ekran paylaşımına geçmeden önce yazı tipini büyütün: **Araçlar > Ayarlar > Düzenleyici > Yazı tipi boyutu = 16**.

Gri başlıklı ve kodu gizli hücreler (form hücreleri) altyapı kodudur; çalıştırmanız yeterli. Katılımcılara göstermeniz gerekmez.

In [ ]:
#@title Kurulum: kütüphaneler ve GPU kontrolü (1–2 dakika)
!pip -q install rank_bm25 pypdf bitsandbytes accelerate

import os, re, time, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU hazır:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU yok. Menüden Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçip yeniden başlatın.")

In [ ]:
#@title Kullanıcılar ve değerlendirme soruları — çalıştırın
# Erdağı Bank kurgusaldır. Dokümanların kendisi GitHub'daki PDF'lerdir (erdagi_bank/ klasörü).

# Eğitimde kullanılan örnek kullanıcı profilleri (yetki demoları için)
USERS = {
    "Şube çalışanı (Elif)":        ["tum_personel"],
    "Kredi tahsis uzmanı (Murat)": ["tum_personel", "kredi_tahsis"],
    "Genel müdür yardımcısı (Can)": ["tum_personel", "kredi_tahsis", "ust_yonetim"],
}

# ---------------------------------------------------------------------
# DEĞERLENDİRME SETİ (golden set)
# relevant: cevabın bulunduğu bölüm(ler) — "doc_id::Madde X" biçiminde
# relevant boş ise soru dokümanlarda cevaplanamaz (sistem "bulamadım" demeli)
# ---------------------------------------------------------------------
GOLDEN_SET = [
 {"q": "150 bin TL'lik ihtiyaç kredisi için hangi belgeler istenir?", "relevant": ["krd-yon-v3::Madde 4.2"],
  "ref": "100.000 TL üzerindeki ihtiyaç kredilerinde gelir belgesi zorunludur (bordro, emekli maaş dökümü veya vergi levhası ve beyanname)."},
 {"q": "Bireysel Kredi Yönergesi Madde 4.3 ne diyor?", "relevant": ["krd-yon-v3::Madde 4.3"],
  "ref": "Erken ödeme tazminatı: kalan vade 36 ay ve altında %1, üzerinde %2."},
 {"q": "Emekli bir müşteri gelir belgesi yerine ne sunabilir?", "relevant": ["krd-yon-v3::Madde 4.2"],
  "ref": "Son ay maaş dökümü; emekli maaş dökümü gelir belgesi yerine geçer."},
 {"q": "Kredimi vaktinden önce kapatırsam ceza öder miyim?", "relevant": ["krd-yon-v3::Madde 4.3"],
  "ref": "Evet, erken ödeme tazminatı alınır: kalan vade 36 aya kadar %1, üzeri %2."},
 {"q": "Taşıt kredisinde en uzun vade kaç ay?", "relevant": ["krd-yon-v3::Madde 4.1"],
  "ref": "Taşıt kredisinde vade 12 ile 48 ay arasındadır; en uzun 48 ay."},
 {"q": "Taksitini 90 günden fazla ödemeyen müşteriye ne olur?", "relevant": ["krd-yon-v3::Madde 5.1"],
  "ref": "Kredi Hukuk Müdürlüğüne devredilir ve yasal takip başlatılır."},
 {"q": "Müşteri kartını kaybettiğini söylüyor, ne yapmalıyım?", "relevant": ["kart-pro::Madde 2"],
  "ref": "Kart derhal kapatılır ve otomatik yeni kart talebi oluşturulur."},
 {"q": "Kart aidatı ne zaman tahsil edilir?", "relevant": ["kart-pro::Madde 3"],
  "ref": "Kartın yıllık yenilenme tarihinde hesap özetine yansıtılarak."},
 {"q": "Yıllık kart ücreti iade talebi kaç günde sonuçlanır?", "relevant": ["kart-pro::Madde 3"],
  "ref": "Talep tarihinden itibaren 30 gün içinde."},
 {"q": "Kartla yapılan bir harcamaya ne kadar süre içinde itiraz edilebilir?", "relevant": ["kart-pro::Madde 4"],
  "ref": "İşlem tarihinden itibaren 120 gün içinde."},
 {"q": "Asgari ödeme tutarı nasıl belirlenir?", "relevant": ["kart-pro::Madde 6"],
  "ref": "Limiti 50.000 TL'ye kadar kartlarda dönem borcunun %20'si, üzerinde %40'ı."},
 {"q": "Gece saat 2'de başka bir bankaya para gönderebilir miyim?", "relevant": ["eft-yon::Madde 2", "eft-yon::Madde 3"],
  "ref": "Evet, FAST ile 7/24 ve tek seferde 100.000 TL'ye kadar; EFT ise yalnızca iş günü 09:00–17:00."},
 {"q": "Şirket hesabından günde ne kadar para transferi yapılabilir?", "relevant": ["eft-yon::Madde 5"],
  "ref": "Sözleşmede tanımlanan tutar; tanımlı değilse varsayılan 2.000.000 TL."},
 {"q": "Bireysel müşterinin mobil bankacılık günlük transfer limiti nedir?", "relevant": ["eft-yon::Madde 4"],
  "ref": "250.000 TL; şubeden 1.000.000 TL'ye kadar artırılabilir."},
 {"q": "KMH faizi nasıl hesaplanır?", "relevant": ["mev-hes::Madde 3"],
  "ref": "Kullanılan tutar üzerinden günlük hesaplanır, aylık hesap kesim tarihinde tahakkuk eder."},
 {"q": "Vadeli hesabımı vadesinden önce bozarsam ne olur?", "relevant": ["mev-hes::Madde 2"],
  "ref": "Faiz işletilmez, yalnızca anapara ödenir."},
 {"q": "Yıllardır hiç işlem yapılmayan hesaplara ne oluyor?", "relevant": ["mev-hes::Madde 5"],
  "ref": "10 yıl hareketsiz kalan hesaplar zamanaşımına uğrar ve TMSF'ye devredilir."},
 {"q": "Gayrimenkul ipoteğinde teminat değeri nasıl hesaplanır?", "relevant": ["tic-teminat::Madde 2"],
  "ref": "Güncel ekspertiz değerinin %75'i."},
 {"q": "20 milyon TL'lik ticari krediyi kim onaylar?", "relevant": ["tic-teminat::Madde 4"],
  "ref": "Genel Müdürlük Kredi Komitesi."},
 {"q": "ChatGPT'ye müşteri bilgisi yazabilir miyim?", "relevant": ["bilgi-guv::Madde 3"],
  "ref": "Hayır; yalnızca BT onaylı iç yapay zekâ asistanı kullanılabilir."},
 {"q": "Şifremi ne sıklıkla değiştirmem gerekiyor?", "relevant": ["bilgi-guv::Madde 1"],
  "ref": "90 günde bir."},
 {"q": "Babalık izni kaç gün?", "relevant": ["ik-izin::Madde 3"], "ref": "5 gün."},
 {"q": "8 yıldır çalışan birinin yıllık izni kaç gündür?", "relevant": ["ik-izin::Madde 1"], "ref": "20 gün."},
 {"q": "Evden çalışma hakkım var mı?", "relevant": ["ik-izin::Madde 4"],
  "ref": "Genel müdürlükte yönetici onayıyla haftada en fazla 2 gün; şube personeli için yok."},
 # --- dokümanlarda cevabı OLMAYAN sorular ---
 {"q": "Erdağı Bank'ın 2025 yılı net kârı ne kadar?", "relevant": [], "ref": "Dokümanlarda yok."},
 {"q": "Konut kredisinde faiz oranı yüzde kaç?", "relevant": [], "ref": "Dokümanlarda yok."},
 {"q": "Yurt dışına SWIFT ile para gönderim ücreti nedir?", "relevant": [], "ref": "Dokümanlarda yok."},
]

# ---------------------------------------------------------------------
# MEVZUAT (kamuya açık) — 3. gün atölyesi
# Kaynak: mevzuat.gov.tr. Bağlantıları eğitimden önce kontrol edin.
# ---------------------------------------------------------------------
MEVZUAT = {
    "kvkk": {"title": "6698 sayılı Kişisel Verilerin Korunması Kanunu",
             "url": "https://www.mevzuat.gov.tr/MevzuatMetin/1.5.6698.pdf"},
    "bk":   {"title": "5411 sayılı Bankacılık Kanunu",
             "url": "https://www.mevzuat.gov.tr/MevzuatMetin/1.5.5411.pdf"},
}

MEVZUAT_GOLDEN = [
 {"q": "Kişisel verilerin işlenmesinde uyulması gereken genel ilkeler nelerdir?", "relevant": ["kvkk::Madde 4"]},
 {"q": "Kişisel veriler hangi durumlarda açık rıza alınmadan işlenebilir?",      "relevant": ["kvkk::Madde 5"]},
 {"q": "Özel nitelikli kişisel veriler nelerdir?",                                "relevant": ["kvkk::Madde 6"]},
 {"q": "Kişisel veriler ne zaman silinmeli veya anonim hale getirilmeli?",        "relevant": ["kvkk::Madde 7"]},
 {"q": "Kişisel veriler yurt dışına hangi şartlarla aktarılabilir?",               "relevant": ["kvkk::Madde 9"]},
 {"q": "Veri sorumlusu kişisel verileri toplarken ilgili kişiye neleri bildirmeli?", "relevant": ["kvkk::Madde 10"]},
 {"q": "Bir müşteri bankadan hakkında hangi bilgileri isteyebilir, hakları nelerdir?", "relevant": ["kvkk::Madde 11"]},
 {"q": "Veri güvenliği için veri sorumlusunun yükümlülükleri nelerdir?",           "relevant": ["kvkk::Madde 12"]},
 {"q": "Veri sahibinin başvurusu en geç kaç gün içinde sonuçlandırılmalıdır?",     "relevant": ["kvkk::Madde 13"]},
 {"q": "Başvurusu reddedilen kişi Kurula ne zaman şikâyette bulunabilir?",         "relevant": ["kvkk::Madde 14"]},
 {"q": "Açık rıza nasıl tanımlanmıştır?",                                          "relevant": ["kvkk::Madde 3"]},
 {"q": "Banka çalışanları müşteri sırlarını kimlere açıklayamaz, sır saklama yükümlülüğü nedir?", "relevant": ["bk::Madde 73"]},
]

print(f"{len(GOLDEN_SET)} değerlendirme sorusu, {len(MEVZUAT_GOLDEN)} mevzuat sorusu yüklendi.")

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"   # çok dilli, Türkçe destekli, açık kaynak; kurum içinde çalıştırılabilir
embedder = SentenceTransformer(
    EMBED_MODEL, device=DEVICE,
    model_kwargs={"torch_dtype": torch.float16} if DEVICE == "cuda" else {},
)
embedder.max_seq_length = 512

def embed(texts):
    """Metin listesini normalize edilmiş vektörlere çevirir (uzunluk = 1)."""
    return embedder.encode(list(texts), normalize_embeddings=True, batch_size=32, show_progress_bar=False)

v = embed(["Merhaba Erdağı Bank"])
print("Model yüklendi. Bir vektörün boyutu:", v.shape[1])

---
# ▶ CANLI DEMO 1.1 — Token ve embedding
**Sunudaki yeri:** M2 (Embedding ve vektörler) sonunda, "CANLI DEMO 1.1" slaytı.

**Amaç:** Modelin metni nasıl parçaladığını ve anlamı nasıl sayıya çevirdiğini göstermek.

**Akış:**
1. Token hücresini çalıştırın. Aynı cümlenin Türkçede İngilizceden daha fazla token tuttuğunu gösterin. Son satırdaki uzun kelimenin nasıl parçalandığına dikkat çekin.
2. Benzerlik oyunu: **Hücreyi çalıştırmadan önce** beş çifti ekranda okuyun ve katılımcılara sorun: *"Sizce hangi çift en benzer, hangisi en az benzer?"* Sonra çalıştırın.
   - Beklenen ilginç sonuç: 2. çift aynı kelimeleri tekrar ettiği halde 1. çift kadar yüksek çıkmayabilir; 1. çift ortak kelime içermediği halde yüksek çıkar. Mesaj: *embedding kelimeyi değil anlamı karşılaştırır.*
   - 4. çift ("banka faiz" ve "parktaki bank") aynı kelime kökünü taşır ama anlam farklıdır.
3. Harita hücresi: 1024 boyutlu vektörleri 2 boyuta indirip çiziyoruz. Üç konunun üç ayrı küme oluşturduğunu gösterin. Tablo, aynı bilginin sayısal halidir: aynı konudaki cümlelerin benzerliği yüksek.

In [ ]:
tokenizer = embedder.tokenizer
ornekler = [
    "The bank increased interest rates on consumer loans.",
    "Banka bireysel kredilerde faiz oranlarını artırdı.",
    "Kredilendirilemeyeceklerimizden",
]
for s in ornekler:
    parcalar = tokenizer.tokenize(s)
    print(f"{len(parcalar):>2} token | {s}")
    print("          ", parcalar, "\n")

In [ ]:
# Önce katılımcılara sorun: Hangi çift en benzer, hangisi en az benzer?
ciftler = [
    ("Kart aidatı ne zaman alınır?",          "Yıllık kart ücreti yenileme tarihinde tahsil edilir."),
    ("Kart aidatı ne zaman alınır?",          "Kartın aidatı aidat ödeme aidat kart."),
    ("Kredimi erken kapatırsam ceza var mı?", "Vadesinden önce kapatılan kredilerde erken ödeme tazminatı alınır."),
    ("Banka faiz oranını artırdı.",           "Parktaki bank kırılmış."),
    ("Şifremi unuttum.",                      "Kredi kartı borcumu nasıl yapılandırırım?"),
]
for a, b in ciftler:
    va, vb = embed([a, b])
    print(f"{float(va @ vb):.3f}   {a}  <->  {b}")

In [ ]:
from sklearn.decomposition import PCA

cumleler = {
    "Kredi":      ["İhtiyaç kredisi için gelir belgesi gerekir.", "Kredinin vadesi 36 aydır.", "Erken ödemede tazminat alınır."],
    "Kart":       ["Kart aidatı yılda bir alınır.", "Kaybolan kart hemen kapatılır.", "Harcamaya 120 gün içinde itiraz edilir."],
    "İK":         ["Babalık izni 5 gündür.", "Yıllık izin talebi İK portalından yapılır.", "Uzaktan çalışma haftada 2 gündür."],
}
metinler = [c for grup in cumleler.values() for c in grup]
renkler  = [g for g, grup in cumleler.items() for _ in grup]
V = embed(metinler)

# 1024 boyutu görebilmek için 2 boyuta indiriyoruz (yalnızca görselleştirme amaçlı)
xy = PCA(n_components=2).fit_transform(V)
plt.figure(figsize=(9, 6))
for g in cumleler:
    idx = [i for i, r in enumerate(renkler) if r == g]
    plt.scatter(xy[idx, 0], xy[idx, 1], s=80, label=g)
    for i in idx:
        plt.annotate(metinler[i][:32], xy[i], fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.legend(); plt.title("Anlamca benzer cümleler vektör uzayında yan yana düşer"); plt.show()

# Benzerlik matrisi
S = V @ V.T
display(pd.DataFrame(S, index=[m[:25] for m in metinler], columns=range(1, len(metinler) + 1)).round(2))

---
# ▶ CANLI DEMO 1.2 — Gerçek mevzuat PDF'i: ayrıştırma ve chunking
**Sunudaki yeri:** M3 (Ingestion) sonunda, "CANLI DEMO 1.2" slaytı.

**Amaç:** Gerçek bir PDF'in ham halini, temizliğin neden gerektiğini ve iki farklı parçalama yönteminin farkını göstermek.

**Akış:**
1. KVKK PDF'ini indirip ham metni gösterin. Satır kırılmalarına, sayfa numaralarına ve satır sonunda bölünmüş kelimelere dikkat çekin. *"Model bu metni okuyacak; ne kadar temizse o kadar iyi."*
2. Temizlik hücresi: kaç karakter kazandığımızı gösterin.
3. Madde sınırları: Mevzuatın "MADDE 5 –" kalıbını kullanarak maddeleri buluyoruz. Bu, yapı-duyarlı parçalamanın temelidir.
4. Karşılaştırma: Sabit boyutlu bölmenin bir parçayı iki maddenin ortasından kestiğini, madde bazlı bölmenin ise maddeyi bütün tuttuğunu gösterin.

**İnternet erişimi notu:** mevzuat.gov.tr Colab'dan erişimi engellerse PDF'leri önceden bilgisayarınıza indirin ve sol menüdeki **Dosyalar** panelinden `kvkk.pdf` ve `bk.pdf` adlarıyla yükleyin. Bunu eğitimden önce mutlaka deneyin.

In [ ]:
import requests
from pypdf import PdfReader

def pdf_indir(url, dosya):
    """Mevzuat PDF'ini indirir. İnternet erişimi engelliyse dosyayı elle yükleyin."""
    if os.path.exists(dosya):
        return dosya
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("İndirilen dosya PDF değil")
    with open(dosya, "wb") as f:
        f.write(r.content)
    return dosya

def pdf_metni(dosya):
    return "\n".join(sayfa.extract_text() or "" for sayfa in PdfReader(dosya).pages)

# --- demo ---
MEVZUAT_DOSYA = {}
for kod, bilgi in MEVZUAT.items():
    dosya = f"{kod}.pdf"
    try:
        MEVZUAT_DOSYA[kod] = pdf_indir(bilgi["url"], dosya)
        print("İndirildi:", bilgi["title"])
    except Exception as e:
        print(f"İndirilemedi ({bilgi['title']}): {e}")
        print(f"  -> Sol menüdeki Dosyalar panelinden '{dosya}' adıyla yükleyip bu hücreyi tekrar çalıştırın.")

ham_kvkk = pdf_metni(MEVZUAT_DOSYA["kvkk"])
print(f"\nKVKK metni: {len(ham_kvkk):,} karakter")
print("=" * 60, "\nHAM METİN (ilk 1200 karakter):\n", ham_kvkk[:1200])

In [ ]:
def metni_temizle(t):
    t = t.replace("\u00ad", "")                       # görünmez tire
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)            # satır sonunda bölünmüş kelimeleri birleştir
    satirlar = [s.strip() for s in t.split("\n")]
    satirlar = [s for s in satirlar if s and not re.fullmatch(r"\d{1,3}", s)]   # tek başına sayfa numaraları
    t = "\n".join(satirlar)
    return re.sub(r"[ \t]+", " ", t)

# --- demo ---
kvkk = metni_temizle(ham_kvkk)
print(f"Temizlik öncesi {len(ham_kvkk):,} karakter, sonrası {len(kvkk):,} karakter")

In [ ]:
# Mevzuatta her madde "MADDE 5 –" biçiminde başlar. Bu kalıbı madde sınırlarını bulmak için kullanıyoruz.
MADDE_RE = re.compile(r"(GEÇİCİ\s+MADDE|Geçici\s+Madde|EK\s+MADDE|Ek\s+Madde|MADDE|Madde)\s+(\d+)\s*\.?\s*[-–—]")
ETIKET = {"MADDE": "Madde", "GEÇİCİ MADDE": "Geçici Madde", "EK MADDE": "Ek Madde"}

def madde_araliklari(metin, kod):
    """Her maddenin metindeki başlangıç ve bitiş konumunu döndürür."""
    bulunan = list(MADDE_RE.finditer(metin))
    araliklar = []
    for i, m in enumerate(bulunan):
        tur = ETIKET.get(re.sub(r"\s+", " ", m.group(1)).upper(), "Madde")
        bas = m.start()
        # Madde başlığı (ör. "Amaç") genellikle bir önceki kısa satırdadır; onu da maddeye dahil et
        onceki = metin.rfind("\n", 0, bas - 1)
        satir = metin[onceki + 1:bas].strip()
        if 0 < len(satir) < 70 and not satir.endswith((".", ":", ";")):
            bas = onceki + 1
        bit = bulunan[i + 1].start() if i + 1 < len(bulunan) else len(metin)
        araliklar.append({"key": f"{kod}::{tur} {m.group(2)}", "etiket": f"{tur} {m.group(2)}", "start": bas, "end": bit})
    # bir sonraki maddenin başlık satırı önceki maddenin sonuna taşmasın
    for a, b in zip(araliklar, araliklar[1:]):
        a["end"] = min(a["end"], b["start"])
    return araliklar

# --- demo ---
kvkk_maddeler = madde_araliklari(kvkk, "kvkk")
print(len(kvkk_maddeler), "madde bulundu. Örnek:")
for a in kvkk_maddeler[10:12]:
    print("-" * 60); print(a["key"]); print(kvkk[a["start"]:a["end"]][:500])

In [ ]:
def token_pencereleri(metin, boyut, ortusme):
    """Metni 'boyut' token'lık, 'ortusme' token örtüşen pencerelere böler; karakter aralıkları döndürür."""
    try:
        ofs = embedder.tokenizer(metin, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    except Exception:
        ofs = [(m.start(), m.end()) for m in re.finditer(r"\S+", metin)]
    if len(ofs) <= boyut:
        return [(0, len(metin))]
    adim, out = max(1, boyut - ortusme), []
    for i in range(0, len(ofs), adim):
        w = ofs[i:i + boyut]
        out.append((w[0][0], w[-1][1]))
        if i + boyut >= len(ofs):
            break
    return out

def chunk_sabit(metin, kod, baslik, boyut=300, ortusme=50):
    """YÖNTEM 1 — Sabit boyut: metnin yapısına bakmadan her N token'da bir böler."""
    maddeler = madde_araliklari(metin, kod)
    parcalar = []
    for j, (s, e) in enumerate(token_pencereleri(metin, boyut, ortusme)):
        ilgili = {a["key"] for a in maddeler if min(e, a["end"]) - max(s, a["start"]) >= 80}
        parcalar.append({"id": f"{kod}#sabit{j}", "text": metin[s:e], "articles": ilgili,
                         "meta": {"doc_id": kod, "title": baslik, "section": f"Parça {j}", "status": "yururlukte"}})
    return parcalar

def chunk_madde(metin, kod, baslik, boyut=300, ortusme=50):
    """YÖNTEM 2 — Yapı-duyarlı: her madde ayrı parça; çok uzun maddeler kendi içinde bölünür.
    Her parçanın başına doküman ve madde adı (başlık zinciri) eklenir."""
    parcalar = []
    for a in madde_araliklari(metin, kod):
        govde = metin[a["start"]:a["end"]].strip()
        for j, (s, e) in enumerate(token_pencereleri(govde, boyut, ortusme)):
            parcalar.append({"id": f"{a['key']}#{j}", "text": f"{baslik} > {a['etiket']}\n{govde[s:e]}",
                             "articles": {a["key"]},
                             "meta": {"doc_id": kod, "title": baslik, "section": a["etiket"], "status": "yururlukte"}})
    return parcalar

In [ ]:
sabit = chunk_sabit(kvkk, "kvkk", MEVZUAT["kvkk"]["title"], boyut=200, ortusme=0)
madde = chunk_madde(kvkk, "kvkk", MEVZUAT["kvkk"]["title"], boyut=400, ortusme=50)
print(f"Sabit boyut: {len(sabit)} parça | Madde bazlı: {len(madde)} parça\n")

# Sabit boyutlu bölmede birden fazla maddeye yayılan bir parça bulalım
for p in sabit:
    if len(p["articles"]) >= 2:
        print("SABİT BOYUT — bu parça", sorted(p["articles"]), "maddelerini ortasından kesiyor:\n")
        print(p["text"][:700]); break

print("\n" + "=" * 60 + "\nMADDE BAZLI — aynı içerik, kendi maddesiyle bütün halinde:\n")
print(next(p for p in madde if "kvkk::Madde 11" in p["articles"])["text"][:700])

---
# ▶ UYGULAMA 1 — Erdağı Bank: parçala, vektörleştir, ara
**Sunudaki yeri:** 1. günün son bloğu, "UYGULAMA 1" slaytı. Yaklaşık 40 dakika.

**Amaç:** Bir RAG sisteminin indeksleme yarısını uçtan uca kurmak ve ilk aramaları yapmak. Gün sonunda dokümanlar vektör veritabanında ve sorgulanabilir olacak.

**Akış:**
1. **Dokümanları göster ve çek:** Önce tarayıcıda GitHub reposundaki `erdagi_bank` klasörünü açıp birkaç PDF'i gösterin: *"Bunlar bir bankanın iç yönergeleri; eski sürüm de, gizli doküman da var."* Sonra hücreyi çalıştırıp PDF'leri ve `manifest.json` dosyasını Colab'a indirin. Doküman listesi tablosunu ve bir PDF'in önizlemesini gösterin.
2. **PDF'ten metne:** Eski sürüm yönergenin ham metninde üstbilgi, altbilgi, doküman kodu ve *"YÜRÜRLÜKTEN KALKMIŞTIR"* filigranının metne karıştığını gösterin. Temizlenmiş halde yalnızca madde başlıkları ve gövdeler kalır. Vurgu: *"Yürürlük durumu, gizlilik ve izinli gruplar PDF'in içinden değil, doküman yönetim sistemini temsil eden manifest dosyasından gelir."*
3. **Parçalama:** Erdağı Bank dokümanlarını başlıklarından bölüyoruz. Ekranda bir parçayı ve metadata'sını gösterin: başlık zinciri ("Bireysel Kredi Yönergesi > Madde 4.2"), yürürlük durumu, izinli gruplar. *"Bu alanlar 3. gün güvenlik konusunda hayat kurtaracak."*
4. **Vektör indeksi:** `VectorIndex` sınıfı en basit vektör veritabanıdır: tüm vektörler bir matriste, arama brute force. Parça sayısı az olduğu için bu yeterli; büyük ölçekte HNSW gibi indeksler devreye girer (bonus bölüm).
5. **İlk arama:** "150 bin liralık" ifadesinin "100.000 TL üzeri" maddesini bulduğunu gösterin; ortak kelime az olduğu halde anlam eşleşiyor.
6. **Metadata filtresi:** Filtresiz aramada eski sürümün (v2, "50.000 TL") da geldiğini gösterin. *"Model bu iki parçayı birlikte görse hangi tutarı söylerdi?"* Ardından filtreli aramada yalnızca yürürlükteki sürümün geldiğini gösterin.
7. **Katılımcı soruları:** Son hücreye katılımcıların önerdiği soruları yazın ve sonuçları birlikte yorumlayın. Doğru bölüm gelmediğinde nedenini birlikte tartışın: ifade farkı mı, kısaltma mı, madde numarası mı? Bu tartışma 2. günün hibrit arama konusuna köprüdür.

In [ ]:
import requests
REPO = "sharpwareco/tbb-rag-egitimi"  #@param {type:"string"}
DAL = "main"
HAM_ADRES = f"https://raw.githubusercontent.com/{REPO}/{DAL}/erdagi_bank"
KLASOR = "erdagi_bank"
os.makedirs(KLASOR, exist_ok=True)

def dosya_getir(ad):
    """Dosyayı GitHub'dan indirir; klasörde zaten varsa (ör. elle yüklendiyse) onu kullanır."""
    yol = os.path.join(KLASOR, ad)
    if not os.path.exists(yol):
        r = requests.get(f"{HAM_ADRES}/{ad}", timeout=60)
        r.raise_for_status()
        with open(yol, "wb") as f:
            f.write(r.content)
    return yol

try:
    MANIFEST = json.load(open(dosya_getir("manifest.json"), encoding="utf-8"))
    for m in MANIFEST:
        dosya_getir(m["file"])
    print(f"{len(MANIFEST)} PDF ve manifest hazır ({KLASOR}/ klasörü)")
except Exception as e:
    print("GitHub'dan indirilemedi:", e)
    print("-> Repodaki erdagi_bank klasörünün içeriğini (manifest.json + PDF'ler) Dosyalar panelinde",
          f"'{KLASOR}' klasörüne yükleyip bu hücreyi tekrar çalıştırın.")

In [ ]:
!pip -q install pypdfium2
import pypdfium2 as pdfium

def pdf_goster(ad, sayfa=0, olcek=1.3):
    """PDF'in bir sayfasını notebook içinde resim olarak gösterir."""
    display(pdfium.PdfDocument(os.path.join(KLASOR, ad))[sayfa].render(scale=olcek).to_pil())

# --- demo ---
display(pd.DataFrame([{"dosya": m["file"], "doküman": m["title"], "sürüm": m["version"], "durum": m["status"],
                       "gizlilik": m["confidentiality"], "izinli gruplar": ", ".join(m["allowed_groups"])}
                      for m in MANIFEST if not m.get("gun3")]))
pdf_goster("krd-yon-v3.pdf")

In [ ]:
# PDF'ten çıkan metinde üstbilgi, altbilgi, sayfa numarası ve filigran da var: bunları ayıklıyoruz
GURULTU = [r"^ERDAĞI BANK$", r"^NOVA BİLİŞİM A\.Ş\.$", r"Doküman kodu:", r"Teklif No:", r"^Sayfa \d+$", r"^\d{1,3}$",
           r"^YÜRÜRLÜKTEN KALKMIŞTIR$",
           r"^(Yönerge|Prosedür|Politika|Yönetmelik|Yönetim Kurulu Kararı|Tedarikçi Teklifi|Şikâyet Kaydı) · "]
BASLIK_RE = re.compile(r"^(Madde \d+(?:\.\d+)?|Bölüm \d+) – .+$")

def pdf_dokuman_oku(meta):
    """PDF'i okur, gürültüyü atar, madde başlıklarını bulur. Metadata PDF'ten değil manifest'ten gelir."""
    satirlar = [s.strip() for s in pdf_metni(os.path.join(KLASOR, meta["file"])).split("\n")]
    satirlar = [s for s in satirlar if s and not any(re.search(g, s) for g in GURULTU)]
    bolumler, aktif = [], None
    for s in satirlar:
        if BASLIK_RE.match(s):
            aktif = [s, []]
            bolumler.append(aktif)
        elif aktif is not None:          # ilk başlıktan önceki kapak bilgisi atılır
            aktif[1].append(s)
    metin = "\n\n".join(f"## {baslik}\n{' '.join(govde)}" for baslik, govde in bolumler)
    return {**meta, "text": metin}

DOCS = [pdf_dokuman_oku(m) for m in MANIFEST]
print(f"{len(DOCS)} doküman PDF'ten okundu.")
# --- demo ---
ornek = next(m for m in MANIFEST if m["doc_id"] == "krd-yon-v2")
print("HAM METİN (eski sürüm yönerge) — üstbilgi, altbilgi ve filigran metne karışmış:\n")
print(pdf_metni(os.path.join(KLASOR, ornek["file"]))[:900])
print("\n" + "=" * 60 + "\nTEMİZLENMİŞ VE BÖLÜMLERİNE AYRILMIŞ:\n")
print(next(d for d in DOCS if d["doc_id"] == "krd-yon-v2")["text"])

In [ ]:
def chunk_dokuman(doc):
    """Erdağı Bank dokümanlarını '## ' başlıklarından böler; her parçaya başlık zinciri ve metadata ekler."""
    parcalar = []
    for bolum in re.split(r"\n(?=## )", doc["text"].strip()):
        satirlar = bolum.strip().split("\n", 1)
        baslik = satirlar[0].lstrip("# ").strip()
        govde = satirlar[1].strip() if len(satirlar) > 1 else ""
        m = re.match(r"(Madde [\d\.]+)", baslik)
        etiket = m.group(1) if m else baslik.split(" – ")[0]
        key = f"{doc['doc_id']}::{etiket}"
        meta = {k: v for k, v in doc.items() if k != "text"}
        meta["section"] = baslik
        parcalar.append({
            "id": key,
            "text": f"{doc['title']} (Sürüm {doc['version']}) > {baslik}\n{govde}",
            "articles": {key},
            "meta": meta,
        })
    return parcalar

def parcala(docs, gun3=False):
    return [p for d in docs if gun3 or not d.get("gun3") for p in chunk_dokuman(d)]

CHUNKS = parcala(DOCS)
print(len(CHUNKS), "parça oluşturuldu.\nÖrnek parça:\n")
print(CHUNKS[7]["text"]); print("\nMetadata:", json.dumps(CHUNKS[7]["meta"], ensure_ascii=False, indent=1))

In [ ]:
def filtreden_gecer(parca, filtre):
    """Metadata filtresi. 'groups' özel anahtardır: kullanıcının gruplarından en az biri dokümanda izinli olmalı."""
    for anahtar, deger in (filtre or {}).items():
        if anahtar == "groups":
            if not set(parca["meta"].get("allowed_groups", [])) & set(deger):
                return False
        elif parca["meta"].get(anahtar) != deger:
            return False
    return True

class VectorIndex:
    """En basit vektör veritabanı: tüm vektörler bellekte, arama brute force (kesin)."""
    def __init__(self, parcalar):
        self.chunks = parcalar
        self.vecs = embed([p["text"] for p in parcalar])

    def search(self, sorgu, k=5, flt=None, sorgu_vektoru=None):
        q = embed([sorgu])[0] if sorgu_vektoru is None else sorgu_vektoru
        skorlar = self.vecs @ q                       # normalize vektörlerde nokta çarpımı = kosinüs benzerliği
        sonuc = []
        for i in np.argsort(-skorlar):
            if filtreden_gecer(self.chunks[i], flt):
                sonuc.append((self.chunks[i], float(skorlar[i])))
                if len(sonuc) == k:
                    break
        return sonuc

def goster(sonuclar):
    """Arama sonuçlarını tablo olarak gösterir."""
    display(pd.DataFrame([{
        "sıra": i, "skor": round(s, 3), "bölüm": p["id"],
        "durum": p["meta"].get("status"), "metin": p["text"].split("\n", 1)[-1][:110],
    } for i, (p, s) in enumerate(sonuclar, 1)]))

t0 = time.time()
vindex = VectorIndex(CHUNKS)
print(f"{len(CHUNKS)} parça vektörleştirildi ({time.time() - t0:.1f} sn). Vektör matrisi: {vindex.vecs.shape}")

In [ ]:
soru = "150 bin liralık ihtiyaç kredisi için hangi belge lazım?"
print("Soru:", soru)
goster(vindex.search(soru, k=5))

In [ ]:
soru = "İhtiyaç kredisinde gelir belgesi hangi tutarın üzerinde zorunlu?"
print("FİLTRESİZ — eski sürüm (v2) de geliyor, iki farklı tutar var:")
goster(vindex.search(soru, k=4))

print("\nFİLTRELİ — yalnızca yürürlükteki dokümanlar:")
goster(vindex.search(soru, k=4, flt={"status": "yururlukte"}))

print("\nFİLTRELİ — yalnızca Kartlar departmanı:")
goster(vindex.search("Müşteri ücret iadesi istiyor", k=3, flt={"department": "kartlar"}))

In [ ]:
#@title Katılımcıların sorusu
SORU = "Kartım çalınırsa ne yapmalıyım?"  #@param {type:"string"}
goster(vindex.search(SORU, k=5, flt={"status": "yururlukte"}))

---
# Bonus — HNSW: büyük ölçekte neden yaklaşık arama?
Zaman kalırsa çalıştırın (indeks kurulumu 1–2 dakika sürer). 100.000 vektörde brute force ile HNSW'yi karşılaştırır. `efSearch` arttıkça doğruluk (ANN recall) artar, hız düşer. Bu, sunudaki "hız için küçük bir doğruluk bedeli" slaytının kanıtıdır.

In [ ]:
# BONUS: Büyük ölçekte neden ANN (HNSW) kullanılır? 100.000 vektörle hız ve doğruluk karşılaştırması
!pip -q install faiss-cpu
import faiss
N, D = 100_000, 1024
rng = np.random.default_rng(0)
# Gerçek embedding'ler 1024 boyutun hepsini eşit kullanmaz; daha düşük boyutlu bir "anlam yüzeyine" yayılır.
# Bunu taklit eden sentetik veri üretiyoruz (48 gizli boyut -> 1024 boyut).
Z = rng.standard_normal((N, 48), dtype=np.float32)
X = Z @ rng.standard_normal((48, D), dtype=np.float32) + 0.5 * rng.standard_normal((N, D), dtype=np.float32)
X /= np.linalg.norm(X, axis=1, keepdims=True)
Q = X[rng.integers(0, N, 50)] + 0.02 * rng.standard_normal((50, D), dtype=np.float32)
Q /= np.linalg.norm(Q, axis=1, keepdims=True)

flat = faiss.IndexFlatIP(D); flat.add(X)
t0 = time.time(); _, I_kesin = flat.search(Q, 10); t_flat = (time.time() - t0) / len(Q)

hnsw = faiss.IndexHNSWFlat(D, 32, faiss.METRIC_INNER_PRODUCT)
t0 = time.time(); hnsw.add(X); print(f"HNSW indeksi {time.time() - t0:.0f} sn'de kuruldu")
for ef in [16, 64, 256]:
    hnsw.hnsw.efSearch = ef
    t0 = time.time(); _, I = hnsw.search(Q, 10); t = (time.time() - t0) / len(Q)
    recall = np.mean([len(set(I[i]) & set(I_kesin[i])) / 10 for i in range(len(Q))])
    print(f"efSearch={ef:>3}: sorgu başı {t*1000:6.2f} ms | ANN recall@10 = {recall:.2f}   (brute force: {t_flat*1000:.2f} ms)")

---
## 1. Günün özeti
- Dokümanları okuduk, temizledik, yapıyı koruyarak parçaladık ve metadata ekledik.
- Parçaları embedding modeliyle vektöre çevirip bir indekse koyduk.
- Anlam bazlı arama yaptık ve metadata filtresinin eski sürümü nasıl dışladığını gördük.

**Yarın:** Bulduğumuz parçaları bir dil modeline verip cevap ürettireceğiz, bunu bir servise dönüştüreceğiz ve kaliteyi ölçeceğiz.